# Deliverable Assembly

**Purpose:** Assemble all final outputs into a clean `deliverable/` folder that can be shared with the supervisor.

**Inputs:** All artifacts from notebooks 01-10
**Output:** `deliverable/` folder containing organised notebooks, tables, figures, README, and summary

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
from pathlib import Path
import shutil, json

REPO = Path(REPO)
DELIVERABLE = REPO / 'deliverable'

# Clean slate
if DELIVERABLE.exists():
    shutil.rmtree(DELIVERABLE)
DELIVERABLE.mkdir()

print(f'Building deliverable at: {DELIVERABLE}')

Building deliverable at: /content/drive/MyDrive/XIDS_Research/xids-research/deliverable


---
## Step 1 — Copy all notebooks

In [ ]:
NOTEBOOKS_OUT = DELIVERABLE / 'notebooks'
NOTEBOOKS_OUT.mkdir()

# Source notebooks in canonical order
notebooks_order = [
    # NSL-KDD pipeline
    '01_data_exploration.ipynb',
    '02_train_models.ipynb',
    '02b_rare_class_boost.ipynb',
    '03_calibration_v2.ipynb',
    '04_shap_computation.ipynb',
    '05_stability_tests.ipynb',
    '06_shap_agreement.ipynb',
    '07_scts_v2.ipynb',
    '08_llm_alerts.ipynb',
    # CIC-IDS2017 pipeline
    '01_cic_data_exploration.ipynb',
    '02_cic_train_models.ipynb',
    '03_cic_calibration.ipynb',
    '04_cic_shap.ipynb',
    '05_cic_stability.ipynb',
    # Final stages
    '09_final_evaluation.ipynb',
    '10_baselines.ipynb',
]

copied = 0
missing = []
for nb in notebooks_order:
    src = REPO / 'notebooks' / nb
    if src.exists():
        shutil.copy(src, NOTEBOOKS_OUT / nb)
        copied += 1
    else:
        missing.append(nb)

print(f'✓ Copied {copied} notebooks')
if missing:
    print(f'⚠ Missing (will be skipped):')
    for m in missing:
        print(f'  - {m}')

---
## Step 2 — Copy paper-ready tables and figures

In [ ]:
TABLES_OUT = DELIVERABLE / 'paper_tables'
TABLES_OUT.mkdir()

PAPER_TABLES_SRC = REPO / 'results' / 'paper_tables'
for f in PAPER_TABLES_SRC.iterdir():
    if f.is_file():
        shutil.copy(f, TABLES_OUT / f.name)

print('Paper tables copied:')
for f in sorted(TABLES_OUT.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

In [ ]:
# All other supporting figures
FIGURES_OUT = DELIVERABLE / 'figures'
FIGURES_OUT.mkdir()

FIGURES_SRC = REPO / 'results' / 'figures'
if FIGURES_SRC.exists():
    for f in FIGURES_SRC.iterdir():
        if f.suffix.lower() in ['.png', '.svg', '.pdf']:
            shutil.copy(f, FIGURES_OUT / f.name)

print(f'Supporting figures copied: {len(list(FIGURES_OUT.iterdir()))} files')

---
## Step 3 — Copy raw tables (CSVs of intermediate results) and LLM alert samples

In [ ]:
RAW_TABLES_OUT = DELIVERABLE / 'raw_tables'
RAW_TABLES_OUT.mkdir()

RAW_SRC = REPO / 'results' / 'tables'
if RAW_SRC.exists():
    for f in RAW_SRC.iterdir():
        if f.suffix == '.csv':
            shutil.copy(f, RAW_TABLES_OUT / f.name)

print(f'Raw tables copied: {len(list(RAW_TABLES_OUT.iterdir()))} files')

In [ ]:
LLM_OUT = DELIVERABLE / 'llm_alerts'
LLM_OUT.mkdir()

LLM_SRC = REPO / 'results' / 'llm_alerts'
if LLM_SRC.exists():
    for f in LLM_SRC.iterdir():
        if f.is_file():
            shutil.copy(f, LLM_OUT / f.name)

print(f'LLM alerts files copied: {len(list(LLM_OUT.iterdir()))} files')

---
## Step 4 — Write README.md and PROJECT_SUMMARY.md

Two markdown files: a full README for the deliverable and a one-page summary for the supervisor.

In [ ]:
# README contents are large — see deliverable_assembly.ipynb companion notebook for full text
# This cell pastes the README content directly

README_CONTENT = '''# Calibrated and Stability-Aware Explainable Intrusion Detection

A Multi-Model SHAP Framework for SOC Decision Support

**Author:** Md Anas Biswas
**Affiliation:** MSc Researcher, University of Portsmouth
**Date:** May 2026

---

## Project overview

This research develops an explainable intrusion detection framework that combines model calibration, explanation stability, conformal prediction, and natural-language alerts into a unified trust score (SCTS-v2) for SOC analysts. The framework is evaluated on two benchmark datasets across three model architectures.

### Five contributions

1. **C1 — Calibrated predictions.** Per-class isotonic regression reduces Expected Calibration Error by 47-98% across all six canonical models, on both datasets.
2. **C2 — Cross-model SHAP agreement (Krishna metrics).** Six agreement metrics from Krishna et al. 2022 quantify how RF, XGBoost, and DNN models disagree on feature importance.
3. **C3 — Explanation stability.** SHAP top-10 rankings tested under Gaussian noise, FGSM, and PGD perturbations.
4. **C4 — SCTS-v2 trust score.** Geometric mean of calibration confidence × explanation stability × conformal coverage.
5. **C5 — LLM-augmented alerts.** Llama-3-8B generates SOC alerts; evaluated via LLM-as-judge.

## Folder structure (this deliverable)

```
deliverable/
├── notebooks/          # All 15-16 Jupyter notebooks (run in order)
├── paper_tables/       # 8 paper-ready CSVs + headline figure
├── figures/            # All supporting figures (PNG)
├── raw_tables/         # Intermediate CSVs (input to paper tables)
├── llm_alerts/         # Sample alerts and judge scores
├── README.md           # This file
└── PROJECT_SUMMARY.md  # One-page summary for review
```

## Datasets

- **NSL-KDD:** Official UNB download; 148,517 samples; 122 features (after one-hot); 5-class labels
- **CIC-IDS2017:** Kaggle mirror; 2.83M flows → 200K stratified subsample; 78 features; 5-class mapped

## Headline findings

See `PROJECT_SUMMARY.md` for the one-page narrative.

## Reproducibility

- Environment: Google Colab Pro (T4 GPU)
- Random seed: 42 throughout
- Code available at github.com/anasbiswas1/xids-research

## Limitations

1. Engelen WTMC-2021 corrections not applied to CIC-IDS2017 (future work)
2. GPT-4o validation deferred to paper writing phase
3. UNSW-NB15 and CICIoT-2023 extensions planned for future work
4. Baseline reproductions (Notebook 10) are representative, not exact
5. U2R class on CIC-IDS2017 has only 36 samples — high-variance F1
'''

(DELIVERABLE / 'README.md').write_text(README_CONTENT)
print('✓ README.md written')

In [ ]:
SUMMARY_CONTENT = '''# X-IDS Project — One-Page Summary

**Author:** Md Anas Biswas | **Date:** May 2026 | **Stage:** End of experimental phase

## What was delivered

An end-to-end explainable intrusion detection framework demonstrating five novel contributions, validated on two benchmark datasets (NSL-KDD and CIC-IDS2017), with reproducible code and paper-ready tables.

## Numerical headlines

| Claim | Evidence |
|---|---|
| Calibration reduces ECE by 47-98% across datasets | NSL-KDD: 67-98%; CIC-IDS2017: 47-91% |
| DNN explanations are most stable | Jaccard 0.86-0.92 vs trees 0.55-0.70 |
| Tree-DNN ranking disagreement is architectural | Trees converge on src_bytes (NSL) and Destination Port (CIC); DNN diverges |
| Rare classes amplify SHAP sign disagreement | Sign agreement drops from 0.96 → 0.78 for R2L/U2R |
| SCTS-v2 stratifies prediction accuracy | Top vs bottom quartile gap: 36-57 percentage points |
| Conformal coverage matches nominal levels | 94.9-97.3% at α=0.05 across all models |
| LLM alert quality tracks trust score | Faithfulness 4.02 → 5.00 from low to high SCTS quartile |

## Deliverable contents

- **15-16 Jupyter notebooks** covering preprocessing → training → calibration → SHAP → stability → SCTS-v2 → LLM alerts → final evaluation → baselines
- **8 paper-ready tables** + **1 summary figure**
- **600+ LLM-generated alerts** with judge scores
- **GitHub repository** with full version control

## What is NOT yet done

| Item | Plan |
|---|---|
| Engelen 2021 corrections on CIC-IDS2017 | Future iteration |
| GPT-4o validation | During paper writing |
| UNSW-NB15 + CICIoT-2023 | Future work |
| Full paper draft | Next phase (2-3 weeks) |

## Status

**Experimental foundation: publishable.** The bottleneck from here is writing.
'''

(DELIVERABLE / 'PROJECT_SUMMARY.md').write_text(SUMMARY_CONTENT)
print('✓ PROJECT_SUMMARY.md written')

---
## Step 5 — Verify the deliverable folder + create archive

In [ ]:
print('=== DELIVERABLE STRUCTURE ===\n')

for root, dirs, files in os.walk(DELIVERABLE):
    level = root.replace(str(DELIVERABLE), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in sorted(files):
        size = (Path(root) / file).stat().st_size / 1024
        print(f'{subindent}{file}  ({size:.1f} KB)')

# Compute total size
total_bytes = sum(f.stat().st_size for f in DELIVERABLE.rglob('*') if f.is_file())
print(f'\nTotal deliverable size: {total_bytes/1024/1024:.1f} MB')

In [ ]:
# Create a zip archive for easy sharing
archive_base = REPO / 'xids_deliverable_2026_05_26'
if (Path(str(archive_base) + '.zip')).exists():
    os.remove(str(archive_base) + '.zip')

shutil.make_archive(str(archive_base), 'zip', DELIVERABLE)

archive_path = Path(str(archive_base) + '.zip')
print(f'✓ Archive created: {archive_path}')
print(f'  Size: {archive_path.stat().st_size/1024/1024:.1f} MB')

---
## Step 6 — Commit and push

In [ ]:
os.chdir(str(REPO))
!git add notebooks/11_deliverable_assembly.ipynb
!git add deliverable/
!git status --short
!git commit -m 'Notebook 11: deliverable assembly — README, summary, organised output folder'
!git push

## Done!

The `deliverable/` folder contains everything organised for your supervisor:

1. **Email the supervisor** with the archive `xids_deliverable_2026_05_26.zip` attached, or share the GitHub link to the `deliverable/` folder.
2. **Point them to `PROJECT_SUMMARY.md` first** — it's the one-page version.
3. **Direct them to `paper_tables/` for the headline numbers** and `notebooks/` for the full code.

Suggested email body:

```
Hi [supervisor name],

Submitting the experimental results package for the X-IDS framework. Two notes:

1. PROJECT_SUMMARY.md gives the one-page overview of contributions, headline numbers, and known limitations.
2. README.md has the full project narrative and reproducibility guide.

Headline findings:
- Calibration reduces ECE by 47-98% across both datasets
- SCTS-v2 stratifies accuracy with up to 57-percentage-point gap between trust quartiles
- LLM alert quality scales monotonically with trust score (4.0 → 5.0)
- Tree-vs-DNN architectural divergence in feature rankings replicates across both datasets

Code is also available at github.com/anasbiswas1/xids-research for reproducibility.

Happy to discuss next steps for the paper draft when convenient.

Best,
[name]
```
